In [ ]:
# 🔧 DIAGNOSTIC CELL - Run this first if you get import errors
# This cell checks and fixes package compatibility issues

import subprocess
import sys
import os

def check_and_fix_packages():
    """Check package versions and suggest fixes."""
    print("🔍 Checking package versions...")
    
    try:
        import transformers
        transformers_version = transformers.__version__
        print(f"  ✅ transformers: {transformers_version}")
    except ImportError:
        print("  ❌ transformers: NOT INSTALLED")
        transformers_version = None
    else:
        # Check if version is too old (need 4.30+ for sentence-transformers 3.x)
        try:
            major, minor = map(int, transformers_version.split('.')[:2])
            if major < 4 or (major == 4 and minor < 30):
                print(f"  ⚠️  transformers version {transformers_version} is too old (need 4.30+)")
        except:
            pass
    
    try:
        import torch
        torch_version = torch.__version__
        print(f"  ✅ torch: {torch_version}")
    except ImportError:
        print("  ❌ torch: NOT INSTALLED")
        torch_version = None
    
    # Try importing sentence_transformers with better error handling
    st_version = None
    st_error = None
    try:
        # Suppress ONNX-related import errors by setting environment variable
        os.environ['SENTENCE_TRANSFORMERS_DISABLE_ONNX'] = '1'
        import sentence_transformers
        st_version = sentence_transformers.__version__
        print(f"  ✅ sentence-transformers: {st_version}")
    except ImportError as e:
        st_error = str(e)
        print(f"  ❌ sentence-transformers: IMPORT FAILED")
        print(f"     Error: {st_error[:100]}...")
    
    # Check for ONNX-related errors
    if st_error and ('onnx' in st_error.lower() or 'openvino' in st_error.lower()):
        print("\n⚠️  ONNX/OpenVINO dependency issue detected")
        print("💡 Solution: Install missing dependencies or disable ONNX:")
        print("   Option 1: !pip install onnx onnxruntime")
        print("   Option 2: Set environment variable (already done above)")
    
    # Suggest fixes
    if transformers_version is None or st_version is None:
        print("\n💡 To fix, run in a new cell:")
        print("   !pip install --upgrade transformers sentence-transformers torch")
        if st_error and ('onnx' in st_error.lower()):
            print("   !pip install onnx onnxruntime  # Optional: for ONNX support")
        print("   Then restart the kernel (Kernel → Restart)")
    elif transformers_version and int(transformers_version.split('.')[0]) < 4:
        print("\n💡 transformers version is too old. Run:")
        print("   !pip install --upgrade transformers")
        print("   Then restart the kernel")

check_and_fix_packages()


In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import os
warnings.filterwarnings('ignore')

# Suppress tqdm warning about ipywidgets and fix compatibility issues
os.environ['TQDM_DISABLE'] = '1'
os.environ['TQDM_NOTEBOOK'] = '0'
# Force tqdm to use console mode instead of notebook mode
try:
    import tqdm
    tqdm.tqdm.pandas = lambda *args, **kwargs: None  # Disable pandas integration if causing issues
except:
    pass

# Import core libraries with error handling
try:
    import umap
except ImportError:
    raise ImportError("umap-learn not installed. Install with: pip install umap-learn")

try:
    import hdbscan
except ImportError:
    raise ImportError("hdbscan not installed. Install with: pip install hdbscan")

try:
    import plotly.express as px
except ImportError:
    print("Warning: plotly not installed. Some visualizations may not work.")
    px = None

try:
    import torch
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
except ImportError:
    raise ImportError("PyTorch not installed. Install with: pip install torch")

from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

# Import sentence_transformers with better error handling
# Disable ONNX to avoid optional dependency issues
os.environ['SENTENCE_TRANSFORMERS_DISABLE_ONNX'] = '1'

try:
    # Try importing transformers first to check if it's available
    try:
        import transformers
    except ImportError:
        raise ImportError(
            "transformers package not installed or broken.\n"
            "Try: pip install --upgrade transformers"
        )
    
    # Now try importing sentence_transformers
    # Suppress warnings about missing ONNX dependencies
    import warnings
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', category=UserWarning)
        warnings.filterwarnings('ignore', category=ImportWarning)
        try:
            from sentence_transformers import SentenceTransformer
            SENTENCE_TRANSFORMERS_AVAILABLE = True
        except ImportError as import_err:
            # If it's an ONNX-related error, try to work around it
            error_str = str(import_err).lower()
            if 'onnx' in error_str or 'openvino' in error_str:
                print("⚠️  ONNX-related import error detected. Trying workaround...")
                # Try to patch the import
                try:
                    import sentence_transformers
                    # Force disable ONNX backend
                    if hasattr(sentence_transformers, 'backend'):
                        sentence_transformers.backend = None
                    from sentence_transformers import SentenceTransformer
                    SENTENCE_TRANSFORMERS_AVAILABLE = True
                    print("✅ Workaround successful - ONNX features disabled")
                except:
                    raise ImportError(
                        "sentence-transformers import failed due to ONNX dependency.\n"
                        "Try: pip install onnx onnxruntime\n"
                        "Or: pip install --upgrade --force-reinstall sentence-transformers"
                    ) from import_err
            else:
                raise
    
except ImportError as e:
    error_msg = str(e)
    SENTENCE_TRANSFORMERS_AVAILABLE = False
    print(f"⚠️ Warning: sentence-transformers import failed: {error_msg}")
    print("\n💡 To fix this, try:")
    print("  1. pip install --upgrade transformers sentence-transformers")
    print("  2. If ONNX error: pip install onnx onnxruntime")
    print("  3. Or: pip uninstall transformers sentence-transformers && pip install transformers sentence-transformers")
    print("  4. Restart Jupyter kernel after installation")
    SentenceTransformer = None

# Import optional packages with error handling
try:
    from keybert import KeyBERT
    KEYBERT_AVAILABLE = True
except ImportError:
    print("Warning: keybert not installed. Install with: pip install keybert")
    KEYBERT_AVAILABLE = False
    KeyBERT = None

try:
    from wordcloud import WordCloud
    WORDCLOUD_AVAILABLE = True
except ImportError:
    print("Warning: wordcloud not installed. Install with: pip install wordcloud")
    WORDCLOUD_AVAILABLE = False
    WordCloud = None
# Print status
if SENTENCE_TRANSFORMERS_AVAILABLE:
    print(f"✅ All required libraries loaded successfully")
    print(f"Using device: {device}")
else:
    print("❌ sentence-transformers not available. Embedding generation will fail.")


In [ ]:
# Load dataset - adjust path if needed
import os
dataset_path = None

# Try to find final_dataset_Preprocessed.csv in organized data structure
alternative_paths = [
    'data/processed/final_dataset_Preprocessed.csv',
    '../data/processed/final_dataset_Preprocessed.csv',
    '../../data/processed/final_dataset_Preprocessed.csv',
    'models/datasets/final_dataset_Preprocessed.csv',  # Legacy path
    '../datasets/final_dataset_Preprocessed.csv',  # Legacy path
    'final_dataset_Preprocessed.csv'  # Current directory
]

for path in alternative_paths:
    if os.path.exists(path):
        dataset_path = path
        break

if dataset_path is None:
    raise FileNotFoundError(
        f"Could not find final_dataset_Preprocessed.csv. Please ensure the file exists in one of these locations:\n"
        f"  - data/processed/final_dataset_Preprocessed.csv (recommended)\n"
        f"  - {alternative_paths}"
    )

df = pd.read_csv(dataset_path)


In [ ]:
df.drop("Heading", axis=1, inplace=True)
df.drop("URL", axis=1, inplace=True)

In [ ]:
df.dropna(inplace=True)

In [ ]:
# Generate embeddings with progress bar
if not SENTENCE_TRANSFORMERS_AVAILABLE:
    raise ImportError(
        "sentence-transformers is not available. Cannot generate embeddings.\n"
        "Please install it first:\n"
        "  pip install --upgrade transformers sentence-transformers\n"
        "Then restart the Jupyter kernel."
    )

print("Generating embeddings...")
try:
    embedder = SentenceTransformer('sentence-transformers/all-mpnet-base-v2', device=device)
    # Use show_progress_bar=False to avoid tqdm notebook compatibility issues
    embeddings = embedder.encode(df["Body"].values, show_progress_bar=False, batch_size=32)
    print(f"Embeddings shape: {embeddings.shape}")
except Exception as e:
    raise RuntimeError(
        f"Error generating embeddings: {str(e)}\n"
        "This might be a version compatibility issue.\n"
        "Try: pip install --upgrade transformers sentence-transformers torch\n"
        "Then restart the Jupyter kernel."
    ) from e

In [ ]:
# Save embeddings to organized data structure
import os
embeddings_dir = 'data/embeddings'
os.makedirs(embeddings_dir, exist_ok=True)
np.save(os.path.join(embeddings_dir, 'embeddings_final_dataset_Preprocessed.npy'), embeddings)

In [11]:
# embeddings = np.load(r"C:\Users\Dhruv\Downloads\embeddings_Headings_final_data_Preprocessed.npy")

In [ ]:
print(embeddings.shape)

(0,)

In [ ]:
# Improved UMAP: Use 10D instead of 2D for better accuracy (preserves more information)
# Also create 2D version for visualization
print("Reducing dimensions with UMAP (10D for clustering, 2D for visualization)...")

# 10D for better clustering accuracy
reducer_10d = umap.UMAP(
    n_components=10,      # Increased from 2 to 10 for better accuracy
    n_neighbors=50,       # Reduced from 100 for better local structure
    min_dist=0.1,         # Increased from 0.02 for better separation
    metric='cosine',      # Better for text embeddings
    random_state=42
)
reduced_embeddings = reducer_10d.fit_transform(embeddings)
print(f"10D embeddings shape: {reduced_embeddings.shape}")

# 2D for visualization
reducer_2d = umap.UMAP(n_components=2, n_neighbors=50, min_dist=0.1, metric='cosine', random_state=42)
reduced_embeddings_2d = reducer_2d.fit_transform(embeddings)print(f"2D embeddings shape: {reduced_embeddings_2d.shape}")


ValueError: Expected 2D array, got 1D array instead:
array=[].
Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.

In [ ]:
# Hyperparameter tuning: Find optimal min_cluster_size
print("\n=== Hyperparameter Tuning ===")
min_cluster_sizes = [15, 20, 25, 30, 35, 40, 50, 60]
results = []

for min_size in min_cluster_sizes:
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=min_size,
        min_samples=5,  # Minimum samples in neighborhood
        cluster_selection_method='eom',  # Excess of Mass
        metric='euclidean'
    )
    labels = clusterer.fit_predict(reduced_embeddings)
    
    # Calculate metrics
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_outliers = list(labels).count(-1)
    outlier_rate = n_outliers / len(labels) * 100
    
    # Calculate silhouette score (filter out outliers)
    if n_clusters >= 2 and outlier_rate < 50:
        mask = labels != -1
        if np.sum(mask) > 1:
            try:
                sil_score = silhouette_score(reduced_embeddings[mask], labels[mask])
                db_score = davies_bouldin_score(reduced_embeddings[mask], labels[mask])
            except:
                sil_score = -1
                db_score = 999
        else:
            sil_score = -1
            db_score = 999
    else:
        sil_score = -1
        db_score = 999
    
    results.append({
        'min_cluster_size': min_size,
        'n_clusters': n_clusters,
        'n_outliers': n_outliers,
        'outlier_rate': outlier_rate,
        'silhouette_score': sil_score,
        'davies_bouldin': db_score
    })
    
    print(f"min_cluster_size={min_size:2d}: {n_clusters:2d} clusters, "
          f"{outlier_rate:5.2f}% outliers, Silhouette={sil_score:.4f}, DB={db_score:.4f}")

# Select best parameters
results_df = pd.DataFrame(results)
results_df['score'] = (
    results_df['silhouette_score'] * 0.4 +  # Weight: 40%
    (1 - results_df['davies_bouldin'] / results_df['davies_bouldin'].max()) * 0.3 +  # Weight: 30%
    (1 - results_df['outlier_rate'] / 100) * 0.3  # Weight: 30%
)

best_idx = results_df['score'].idxmax()
best_min_size = int(results_df.loc[best_idx, 'min_cluster_size'])
best_result = results_df.loc[best_idx]

print(f"\n=== Best Parameters ===")
print(f"Optimal min_cluster_size: {best_min_size}")
print(f"Number of clusters: {int(best_result['n_clusters'])}")
print(f"Outlier rate: {best_result['outlier_rate']:.2f}%")
print(f"Silhouette Score: {best_result['silhouette_score']:.4f}")
print(f"Davies-Bouldin Index: {best_result['davies_bouldin']:.4f}")

# Train final model with best parameters
print(f"\n=== Training Final Model ===")
final_clusterer = hdbscan.HDBSCAN(
    min_cluster_size=best_min_size,
    min_samples=5,
    cluster_selection_method='eom',
    metric='euclidean'
)
labels = final_clusterer.fit_predict(reduced_embeddings)
df["label"] = [str(label) for label in labels]

# Calculate final metrics
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_outliers = list(labels).count(-1)
outlier_rate = n_outliers / len(labels) * 100

mask = labels != -1
if np.sum(mask) > 1:
    final_sil_score = silhouette_score(reduced_embeddings[mask], labels[mask])
    final_db_score = davies_bouldin_score(reduced_embeddings[mask], labels[mask])
    final_ch_score = calinski_harabasz_score(reduced_embeddings[mask], labels[mask])
else:
    final_sil_score = -1
    final_db_score = 999
    final_ch_score = 0
print(f"Final Model Results:")
print(f"  Number of clusters: {n_clusters}")
print(f"  Number of outliers: {n_outliers} ({outlier_rate:.2f}%)")
print(f"  Silhouette Score: {final_sil_score:.4f} (higher is better, range: -1 to 1)")
print(f"  Davies-Bouldin Index: {final_db_score:.4f} (lower is better)")
print(f"  Calinski-Harabasz Score: {final_ch_score:.2f} (higher is better)")
print(f"Num of clusters: {labels.max()}")


Num of clusters: 30


In [ ]:
# Put 2D values for visualization (using 2D embeddings created earlier)
df["x"] = reduced_embeddings_2d[:, 0]
df["y"] = reduced_embeddings_2d[:, 1]
# Substring of the full text, for visualization purposes
df["text_short"] = df["Body"].str[:200]

In [ ]:
hover_data = {
    "text_short": True,
    "x": False,
    "y": False
}
fig = px.scatter(df, x="x", y="y", template="plotly_dark",
                   title="Embeddings", hover_data=hover_data)
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
num_outliers = len(df[df["label"] == "-1"])print(f"Num of outliers: {num_outliers} ({num_outliers / len(df) * 100:.2f} % of total)")


Num of outliers: 3456 (29.84 % of total)


In [ ]:
df_no_outliers = df[df["label"] != "-1"]

# scatter plot
hover_data = {
    "text_short": True,
    "x": False,
    "y": False
}
fig = px.scatter(df_no_outliers, x="x", y="y", template="plotly_dark",
                   title="Embeddings", color="label", hover_data=hover_data)
fig.show()

In [ ]:
# Test cluster keyword extraction (using Body since Heading was dropped)
if KEYBERT_AVAILABLE:
    cluster = "1"
    df_subset = df[df["label"] == cluster].reset_index()
    texts_concat = ". ".join(df_subset["Body"].values[:50])  # Use Body, limit to 50 for speed
    keywords_and_scores = KeyBERT().extract_keywords(texts_concat,
                                        keyphrase_ngram_range=(1, 2), top_n=10)
    print(keywords_and_scores)
else:
    print("KeyBERT not available. Install with: pip install keybert")

[('schumacher', 0.5447), ('alonso', 0.5014), ('ferrari', 0.4603), ('prix', 0.4576), ('prixs', 0.4172), ('racing', 0.3931), ('jenson', 0.3871), ('ferraris', 0.375), ('ricciardo', 0.3729), ('fia', 0.368)]


In [ ]:
def filter_keywords(keywords, n_keep=3):
    new_keywords = []
    for candidate_keyword in keywords:
        is_ok = True
        for compare_keyword in keywords:
            if candidate_keyword == compare_keyword:
                continue
            if compare_keyword in candidate_keyword:
                is_ok = False
                break
        if is_ok:
            new_keywords.append(candidate_keyword)
            if len(new_keywords) >= n_keep:
                break
    return new_keywords

keywords = [t[0] for t in keywords_and_scores]
keywords_filtered = filter_keywords(keywords)print(keywords_filtered)


['schumacher', 'alonso', 'ferrari']


In [ ]:
df_no_outliers

In [ ]:
# Assign a meaningful name to each cluster (using Body since Heading was dropped)
def get_cluster_name(df, cluster):
    if not KEYBERT_AVAILABLE:
        return f"Cluster_{cluster}"
    
    df_subset = df[df["label"] == cluster].reset_index()
    texts_concat = ". ".join(df_subset["Body"].values[:100])  # Use Body, limit to 100 for speed
    kw_model = KeyBERT()
    keywords_and_scores = kw_model.extract_keywords(texts_concat, keyphrase_ngram_range=(1, 2),
                                        top_n=10)
    keywords = [t[0] for t in keywords_and_scores]
    keywords_filtered = filter_keywords(keywords)
    return " - ".join(keywords_filtered)

# Get all the new cluster names
all_clusters = df_no_outliers["label"].unique()
d_cluster_name_mapping = {}
print("\n=== Extracting Cluster Names ===")
for cluster in all_clusters:
    if cluster == "-1":
        d_cluster_name_mapping[cluster] = "outliers"
    else:
        cluster_name = get_cluster_name(df_no_outliers, cluster)
        d_cluster_name_mapping[cluster] = cluster_name
        print(f"Cluster {cluster}: {cluster_name}")

# Rename clusters
df_no_outliers["label"] = df_no_outliers["label"].apply(lambda label: d_cluster_name_mapping[label])

In [ ]:
if WORDCLOUD_AVAILABLE:
    clusters = df_no_outliers['label'].unique()
    
    for cluster in clusters:
        # Combine all text for a specific cluster
        cluster_text = ' '.join(df_no_outliers[df_no_outliers['label'] == cluster]['Body'])
        
        # Generate a word cloud for the cluster's text
        wordcloud = WordCloud(width=800, height=400, background_color='white').generate(cluster_text)
        
        # Display the word cloud
        plt.figure(figsize=(10, 5))
        plt.imshow(wordcloud, interpolation='bilinear')
        plt.title(f'Word Cloud for Cluster {cluster}')
        plt.axis('off')
        plt.show()
else:
    print("WordCloud not available. Install with: pip install wordcloud")